# Setup

In [ ]:
!uv pip install torch>=2.0.0
!uv pip install transformers>=4.30.0
!uv pip install datasets>=2.10.0
!uv pip install peft>=0.4.0
!uv pip install accelerate>=0.20.0
!uv pip install bitsandbytes>=0.39.0
!uv pip install PyYAML>=6.0
!uv pip install numpy>=1.21.0
!uv pip install tqdm>=4.64.0
!uv pip install unsloth[colab-new]
!uv pip install xformers

In [ ]:
!uv pip install unsloth
from unsloth import FastLanguageModel

In [ ]:
# Configuration for GPT-OSS20B Cybersecurity Fine-tuning

# Model configuration
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
output_dir = "./gpt-oss-cybersecurity-lora"
max_seq_length = 1024

# Dataset configuration
validation_split = 0.1

# LoRA configuration
lora_r = 16
lora_alpha = 32
lora_dropout = 0.1

# Training configuration
batch_size = 1
eval_batch_size = 2
gradient_accumulation_steps = 16
epochs = 3
learning_rate = 0.0002
logging_steps = 10
eval_steps = 100
save_steps = 500
warmup_steps = 100

# Hardware considerations:
# - Reduce batch_size if you have memory issues
# - Increase gradient_accumulation_steps to maintain effective batch size
# - Consider reducing max_length for memory constraints
# - Use load_in_4bit instead of load_in_8bit for even lower memory usage

# Training

In [ ]:
#!/usr/bin/env python3
"""
Fine-tuning Llama 3.2 for Cybersecurity with Unsloth
This script implements Parameter-Efficient Fine-Tuning (PEFT) using Low-Rank Adaptation (LoRA)
to fine-tune a Llama 3.2 model for cybersecurity applications.
Requirements:
- CUDA-capable GPU with at least 16GB VRAM (recommended)
- Python 3.8+
- See requirements.txt for package dependencies
Usage:
    python fine_tune_gpt_oss_cybersecurity.py --config config.yaml
Date: 2025-08-10
"""

import argparse
import json
import logging
import os
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from datasets import Dataset, load_dataset, concatenate_datasets
from transformers import AutoTokenizer, TrainingArguments
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import yaml



In [ ]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [ ]:
llama3_template = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{}<|eot_id|>"""

def format_prompt_llama3(example: Dict) -> Dict:
    """Formats an example for Llama-3.2-Instruct fine-tuning."""
    instruction = example.get('instruction') or example.get('question') or example.get('input') or ""
    response = example.get('response') or example.get('answer') or example.get('output') or ""
    return {"text": llama3_template.format(instruction, response)}

class CybersecurityFineTuner:
    def __init__(self, config: Dict):
        self.config = config
        self.model_name = config.get('model_name', 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit')
        self.output_dir = config.get('output_dir', './llama3-cybersecurity-lora')
        self.max_length = config.get('max_length', 1024)
        self.tokenizer = None
        self.model = None
        self.train_dataset = None
        self.eval_dataset = None

    def load_model(self) -> None:
        logger.info(f"Loading model from {self.model_name}")
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.model_name,
            max_seq_length=self.max_length,
            dtype=None,
            load_in_4bit=True,
        )
        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=self.config.get('lora_r', 16),
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=self.config.get('lora_alpha', 32),
            lora_dropout=self.config.get('lora_dropout', 0.1),
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=3407,
        )

    def load_and_prepare_datasets(self, downsample_size: int = 5000) -> None:
        dataset_names = [
            "Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset",
            "Cogensec/OptikalLLM_10k_dataset",
            "Mohannadcse/cybersec-reasoning-merged"
        ]
        logger.info(f"Loading datasets: {', '.join(dataset_names)}")
        datasets = [load_dataset(name, split='train') for name in dataset_names]

        def standardize(example):
            # Robustly find instruction and response keys
            instr = example.get('instruction') or example.get('prompt') or ""
            resp = example.get('response') or example.get('output') or ""
            return {"instruction": instr, "response": resp}

        standardized_datasets = []
        for ds in datasets:
            standardized_datasets.append(ds.map(standardize, remove_columns=ds.column_names))

        merged_dataset = concatenate_datasets(standardized_datasets)
        if downsample_size > 0 and len(merged_dataset) > downsample_size:
            merged_dataset = merged_dataset.shuffle(seed=42).select(range(downsample_size))

        self.dataset = merged_dataset.map(format_prompt_llama3)
        if self.config.get('validation_split', 0.1) > 0:
            split = self.dataset.train_test_split(test_size=self.config.get('validation_split', 0.1))
            self.train_dataset, self.eval_dataset = split["train"], split["test"]
        else:
            self.train_dataset, self.eval_dataset = self.dataset, None

    def train(self):
        self.load_model()
        self.load_and_prepare_datasets(downsample_size=self.config.get('downsample_size', 5000))
        trainer = UnslothTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            args=UnslothTrainingArguments(
                per_device_train_batch_size=self.config.get('batch_size', 2),
                gradient_accumulation_steps=self.config.get('gradient_accumulation_steps', 8),
                warmup_steps=self.config.get('warmup_steps', 100),
                num_train_epochs=self.config.get('epochs', 3),
                learning_rate=self.config.get('learning_rate', 2e-4),
                fp16=not torch.cuda.is_bf16_supported(),
                bf16=torch.cuda.is_bf16_supported(),
                logging_steps=self.config.get('logging_steps', 1),
                optim="adamw_8bit",
                output_dir=self.output_dir,
            ),
        )
        trainer.train()

In [ ]:
def main():
    """Main function adapted for notebook execution using global variables."""
    # Build configuration from global variables defined in the notebook
    config = {
        "model_name": model_name,
        "output_dir": output_dir,
        "max_length": max_seq_length,
        "validation_split": validation_split,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "epochs": epochs,
        "learning_rate": learning_rate,
        "logging_steps": logging_steps,
        "warmup_steps": warmup_steps,
        "downsample_size": 5000 # Default for safety, can be moved to config cell if needed
    }

    logger.info("Starting training with notebook-defined configuration")

    fine_tuner = CybersecurityFineTuner(config)
    fine_tuner.train()

if __name__ == "__main__":
    main()

# Inferencing

In [ ]:
import torch
from unsloth import FastLanguageModel
import os

# --- STEP 1: LOAD MODEL & ADAPTERS ---
# This only needs to run once per session
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

checkpoint_dirs = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-') and os.path.isdir(os.path.join(output_dir, d))]
if not checkpoint_dirs:
    raise FileNotFoundError(f"No checkpoint directories found in {output_dir}.")

latest_checkpoint_name = sorted(checkpoint_dirs, key=lambda x: int(x.split('-')[1]))[-1]
latest_checkpoint_path = os.path.join(output_dir, latest_checkpoint_name)

print(f"Loading adapters from: {latest_checkpoint_path}")
model.load_adapter(latest_checkpoint_path)
print("Model and adapters loaded successfully.")

In [ ]:
# --- STEP 2: RUN INFERENCE ---
prompt_text = "Explain what an SQL injection attack is and how to prevent it."
formatted_prompt = llama3_template.format(prompt_text, "")

inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    do_sample=True,
    temperature=1,
    top_k=50,
    top_p=0.95,
    eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],
)

print("Inference complete. Run the next cell to see the result.")

In [ ]:
# --- STEP 3: DECODE & DEBUG ---
# print(f"\n--- Raw Model Outputs ---")
# print(outputs)

response = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

# Extraction logic: Look for the LAST occurrence of the assistant header
# to skip any echoes or template duplicates.
start_marker = "<|start_header_id|>assistant<|end_header_id|>"
end_marker = "<|eot_id|>"

# Find the last start marker
start_index = response.rfind(start_marker)

if start_index != -1:
    # Move index past the marker and any trailing newlines
    content_start = start_index + len(start_marker)

    # Find the end marker specifically after the start content
    end_index = response.find(end_marker, content_start)

    if end_index != -1:
        extracted_response = response[content_start:end_index].strip()
    else:
        extracted_response = response[content_start:].strip()
else:
    extracted_response = f"Could not parse assistant's response. Full text:\n{response}"

print(f"\n--- Inference Result ---")
print(f"Prompt: {prompt_text}")
print(f"Generated Response: {extracted_response}")

## Performance Validation: Fine-tuned vs. Base Model
This section runs a side-by-side comparison to see if the model has actually improved on cybersecurity knowledge.

In [ ]:
import pandas as pd
from unsloth import FastLanguageModel

# Enable native unsloth inference speedups
FastLanguageModel.for_inference(model)

# Expanded and more diverse evaluation prompts
eval_prompts = [
    "What are the primary indicators of a Buffer Overflow vulnerability in C code?",
    "Write a secure Python function to handle user file uploads.",
    "Explain the difference between Stored and Reflected XSS.",
    "How should a SOC analyst respond to a detected Brute Force attack on an SSH service?",
    "Explain the concept of ARP Spoofing and how it can be mitigated at the network layer.",
    "Compare the security of AES and DES encryption algorithms.",
    "Provide an Nmap command to perform a comprehensive service version detection and OS fingerprinting scan."
]

def get_improved_response(model, tokenizer, prompt):
    formatted = llama3_template.format(prompt, "")
    inputs = tokenizer([formatted], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

    marker = "<|start_header_id|>assistant<|end_header_id|>"
    if marker in decoded:
        parts = decoded.split(marker)
        response = parts[-1].replace("<|eot_id|>", "").replace("<|end_of_text|>", "").strip()
        return response
    return decoded.strip()

results = []

print("--- Generating with FINE-TUNED model (Optimized) ---")
for p in eval_prompts:
    results.append({"Prompt": p, "Fine-tuned": get_improved_response(model, tokenizer, p)})

print("--- Generating with BASE model ---")
with model.disable_adapter():
    for i, p in enumerate(eval_prompts):
        results[i]["Base Model"] = get_improved_response(model, tokenizer, p)

df_compare = pd.DataFrame(results)
display(df_compare.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}))

In [ ]:
# Detailed verification of the outputs
for i, res in enumerate(results[:2]): # Checking first two prompts
    print(f"--- PROMPT {i+1}: {res['Prompt']} ---")
    print(f"\n[COLUMN: FINE-TUNED]\n{res['Fine-tuned']}")
    print(f"\n[COLUMN: BASE MODEL]\n{res['Base Model']}")
    print("-" * 50)

# Also check if the adapter is currently active in the model
print(f"Is model currently using an adapter? {model.active_adapter if hasattr(model, 'active_adapter') else 'No active adapter detected'}")

In [ ]:
import json
import matplotlib.pyplot as plt

# Load the trainer state to see the loss curve
stats_path = os.path.join(latest_checkpoint, 'trainer_state.json')

if os.path.exists(stats_path):
    with open(stats_path, 'r') as f:
        stats = json.load(f)

    steps = [log['step'] for log in stats['log_history'] if 'loss' in log]
    losses = [log['loss'] for log in stats['log_history'] if 'loss' in log]

    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, label='Training Loss')
    plt.title('Training Loss Curve')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    print(f"Final recorded loss: {losses[-1] if losses else 'N/A'}")
else:
    print("Trainer state not found. Could not plot loss.")

### Export to GGUF for Ollama
To use this model in Ollama, we need to export it to GGUF format. You can choose different quantization methods (e.g., `q4_k_m` is a good balance of speed and quality).

In [ ]:
# Re-running GGUF export after ensuring config.json exists
from unsloth import FastLanguageModel
import os

# 1. Ensure we are targeting the correct latest checkpoint
checkpoint_dirs = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
latest_checkpoint = os.path.join(output_dir, sorted(checkpoint_dirs, key=lambda x: int(x.split('-')[1]))[-1])

# 2. Load model + adapter correctly
# This time it should detect the PEFT/LoRA weights properly
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = latest_checkpoint,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 3. Export to GGUF
# Using q4_k_m for a good balance of size and performance
model.save_pretrained_gguf(
    "model_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

print(f"Success! Your fine-tuned model from {latest_checkpoint} has been exported to the 'model_gguf' folder.")

In [ ]:
from google.colab import files
import os

# Path to the exported GGUF file
gguf_path = "model_gguf_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf"

if os.path.exists(gguf_path):
    print(f"Downloading {gguf_path}...")
    files.download(gguf_path)
else:
    print("GGUF file not found. Please check the sidebar for the correct path.")

## Fix: Missing config.json

In [ ]:
import os
import shutil
from transformers import AutoConfig

# 1. Check if config.json exists in the checkpoint directory
config_path = os.path.join(latest_checkpoint, 'config.json')

if not os.path.exists(config_path):
    print(f"[!] config.json missing in {latest_checkpoint}. Copying from base model...")
    # Load config from the base model name used in training
    config = AutoConfig.from_pretrained(model_name)
    config.save_pretrained(latest_checkpoint)
    print("[+] config.json has been created in the checkpoint directory.")
else:
    print(f"[+] config.json already exists in {latest_checkpoint}.")

# 2. List directory to verify
print(f"\nContents of {latest_checkpoint}:")
print(os.listdir(latest_checkpoint))

### Re-try GGUF Export
Once the configuration file is verified/fixed above, run the export cell again. The `save_pretrained_gguf` method requires the `config.json` to properly map the architecture for the GGUF file.